# sympy-helpers examples

This notebook demonstrates the public API of **sympy-helpers**:

- `create_mean_symbol`, `create_mean_symbols`
- `create_perturbation_symbol`, `create_perturbation_symbols`
- `create_subscripted_symbol`, `create_subscripted_symbols`
- `display_symbol_dict`, `format_symbol_dict`
- `linearize`
- `get_coefficients_of`

Install the package first from the project root:

```bash
conda activate py310
pip install -e .
```

In [17]:
import sympy as sp
from sympy_helpers import (
    create_mean_symbol,
    create_mean_symbols,
    create_perturbation_symbol,
    create_perturbation_symbols,
    create_subscripted_symbol,
    create_subscripted_symbols,
    display_symbol_dict,
    get_coefficients_of,
    linearize,
)

sp.init_printing()

## Symbol helpers

These functions create related symbols for mean/perturbation analysis and indexed states.
Singular helpers act on one symbol; plural helpers accept a **list** or **mapping** of symbols
and return a dictionary. With mappings, output keys match your logical names (e.g. `"rho"`).

Use `display_symbol_dict` to render those dictionaries with LaTeX values in notebook output.

### Mean symbols

`create_mean_symbol` wraps a symbol name with an overline, preserving assumptions.
Use a backslash prefix for Greek letters (e.g. `\\rho`).

In [18]:
x = sp.Symbol("x", real=True)
rho = sp.Symbol(r"\rho", positive=True)

mean_x = create_mean_symbol(x)
mean_rho = create_mean_symbol(rho)

mean_x, mean_rho

In [19]:
x, y = sp.symbols("x y", real=True)

display_symbol_dict(create_mean_symbols([x, y]))

<IPython.core.display.Math object>

### Symbol mappings

Pass a dictionary when logical names differ from SymPy symbol names, for example in a
thermodynamics/fluid-mechanics variable set:

In [20]:
symbols = {
    "rho": sp.Symbol(r"\rho", real=True, positive=True),  # Density
    "u": sp.Symbol("u", real=True),  # Velocity
    "p": sp.Symbol("p", real=True, positive=True),  # Pressure
    "h": sp.Symbol("h", real=True),  # Enthalpy
    "ht": sp.Symbol("h_t", real=True),  # Total enthalpy
    "qdot": sp.Symbol(r"\dot{q}", real=True),  # Heat release rate (per unit area)
    "T": sp.Symbol("T", real=True),  # Temperature
    "R": sp.Symbol("R", real=True),  # Gas constant
    "cp": sp.Symbol("c_p", real=True),  # Specific heat at constant pressure
    "cv": sp.Symbol("c_v", real=True),  # Specific heat at constant volume
}

display_symbol_dict(create_mean_symbols(symbols))

<IPython.core.display.Math object>

In [21]:
display_symbol_dict(create_perturbation_symbols(symbols))

<IPython.core.display.Math object>

In [22]:
display_symbol_dict(create_subscripted_symbols(symbols, "0"))

<IPython.core.display.Math object>

### Perturbation symbols

`create_perturbation_symbol` appends a suffix (default: `'`) to mark perturbations.
Use `suffix="tilde"` for a LaTeX tilde accent (`\\tilde{x}`).

In [23]:
x = sp.Symbol("x")

perturb_x = create_perturbation_symbol(x)
perturb_x_tilde = create_perturbation_symbol(x, suffix="tilde")

perturb_x, perturb_x_tilde

In [24]:
x, y = sp.symbols("x y")

display_symbol_dict(create_perturbation_symbols([x, y]))

<IPython.core.display.Math object>

### Subscripted symbols

`create_subscripted_symbol` subscripts a single symbol.
`create_subscripted_symbols` returns a dictionary mapping keys to subscripted variants.

In [25]:
x = sp.Symbol("x", real=True)

create_subscripted_symbol(x, "0")

In [26]:
x, y = sp.symbols("x y", real=True)

display_symbol_dict(create_subscripted_symbols([x, y], "0"))

<IPython.core.display.Math object>

## Linearization

`linearize` performs a first-order Taylor expansion around mean values.
By default it returns only perturbation terms (`remove_mean=True`).

### Single-variable example

For `x**2`, the linearized perturbation term is `2*overline(x)*x'`.

In [27]:
x = sp.Symbol("x")
expr = x**2

linearize(expr, [x])

### Including the mean value

Set `remove_mean=False` to retain the expression evaluated at mean values.

In [28]:
x = sp.Symbol("x")
expr = 3 * x + 5

linearize(expr, [x], remove_mean=False)

### Multivariate example

In [29]:
x, y = sp.symbols("x y")
expr = x * y

linearize(expr, [x, y])

### Validation behavior

With `strict=True` (default), missing variables raise `ValueError`.
With `strict=False`, missing variables are ignored silently.

In [30]:
x, y = sp.symbols("x y")
expr = x**2

try:
    linearize(expr, [y], strict=True)
except ValueError:
    print("strict=True raised ValueError for missing variable")

linearize(expr, [y], strict=False)

strict=True raised ValueError for missing variable


/home/calanyalioglu/Projects/sympy_helpers/src/sympy_helpers/linearization.py:43: UserWarning: Variable y not found in expression x**2
  warnings.warn(f"Variable {var} not found in expression {expr}")


## Coefficient extraction

`get_coefficients_of` returns a dictionary mapping each variable to its coefficient in the expanded expression.

In [31]:
x, y = sp.symbols("x y")
expr = 2 * x + 3 * y + 1

display_symbol_dict(get_coefficients_of(expr, [x, y]))

<IPython.core.display.Math object>

In [32]:
x = sp.Symbol("x")
expr = (x + 1) ** 2

display_symbol_dict(get_coefficients_of(expr, [x]))

<IPython.core.display.Math object>